In [ ]:
code = 'CE_BUTTERFLY'
pickle_path = 'C:/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output/'

from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [ ]:
def CE_BUTTERFLY(bt, start_time, end_time, sell_om, buy_om, wing_width, max_rolls, cap_action):
    try:
        start_dt = datetime.datetime.combine(bt.current_week_dates[0], start_time)
        end_dt = datetime.datetime.combine(bt.current_week_dates[-1], end_time)
        
        while True:

            sell_ce_scrip, sell_ce_price, future_price, sell_start_dt = bt.get_strike(start_dt, end_dt, om=sell_om, obove_target_only=True, only='CE')
            if sell_ce_scrip is None: return None

            buy_ce_scrip, buy_ce_price, future_price, buy_start_dt = bt.get_strike(sell_start_dt, end_dt, om=buy_om, obove_target_only=False, only='CE')
            if buy_ce_scrip is None: return None

            if sell_start_dt == buy_start_dt:
                break
            else:
                start_dt = sell_start_dt + datetime.timedelta(minutes=1)

        entry_time = sell_start_dt
        future_price = bt.future_data.loc[entry_time, 'close']
        synthetic_future_data = bt.get_synthetic_future_data()
        synthetic_future_data = synthetic_future_data[(synthetic_future_data.date_time > entry_time) & (synthetic_future_data.date_time <= end_dt)]   # only the live window
        
        sold_strike = get_strike(sell_ce_scrip)
        sold_qnty = 1
        rolls = []
        capped = False
        exit_dt = end_dt
        
        orderbook = [(entry_time, sell_ce_scrip, sell_ce_price, 'SELL', sold_qnty), (entry_time, buy_ce_scrip, buy_ce_price, 'BUY', sold_qnty)]
        
        for row in synthetic_future_data.itertuples():
            
            # action condition
            if row.sync_future > sold_strike:
                
                # max_rolls: -1 = unlimited, 0 = never roll, N = stop after N rolls
                if max_rolls == 0:
                    capped = True
                    if cap_action == "EXIT": exit_dt = row.date_time
                    break
                
                try:
                    time = row.date_time
                    
                    ### scrip traded
                    buy1 = f"{int(sold_strike)}CE"
                    new_sell = f"{int(sold_strike+wing_width)}CE"                
                    buy2 = f"{int(sold_strike+wing_width+wing_width)}CE"

                    ### scrip price
                    buy1_price = bt.options_data.loc[(time, buy1), 'close']
                    new_sell_price = bt.options_data.loc[(time, new_sell), 'close']
                    buy2_price = bt.options_data.loc[(time, buy2), 'close']
                    
                    ###### orders #####
                    orderbook += [(time, buy1, buy1_price, 'BUY', sold_qnty)]
                    orderbook += [(time, new_sell, new_sell_price, 'SELL', sold_qnty+sold_qnty)]
                    orderbook += [(time, buy2, buy2_price, 'BUY', sold_qnty)]

                    ###### roll log (log before the state update) #####
                    rolls.append(f"{time:%d-%b %H:%M} B {int(sold_strike)}x{sold_qnty}@{buy1_price:.2f} S {int(sold_strike+wing_width)}x{sold_qnty*2}@{new_sell_price:.2f} B {int(sold_strike+2*wing_width)}x{sold_qnty}@{buy2_price:.2f}")

                    sold_strike = int(sold_strike + wing_width)
                    sold_qnty = sold_qnty + sold_qnty

                except:
                    pass
                
                if max_rolls > 0 and len(rolls) >= max_rolls:
                    capped = True
                    if cap_action == "EXIT": exit_dt = time
                    break
        
        # ---- square off what's left at exit_dt (cap moment if capped+EXIT, else end of week) (appended to orderbook as normal fills) ----
        pos = {}
        for _, scrip, _, side, qty in orderbook:
            pos[scrip] = pos.get(scrip, 0) + (qty if side == 'BUY' else -qty)
        squareoff = []
        for scrip, qty in pos.items():
            if qty:
                try:
                    exit_price = bt.options_data.loc[(exit_dt, scrip), 'close']
                except KeyError:                                              # no candle at exit_dt -> last one before it
                    exit_price = bt.options_data.xs(scrip, level=1)['close'].loc[:exit_dt].iloc[-1]
                orderbook.append((exit_dt, scrip, exit_price, 'SELL' if qty > 0 else 'BUY', abs(qty)))
                squareoff.append(f"{scrip} {qty:+d}@{exit_price:.2f}")
        
        # ---- pnl = sold - bought - slippage (slippage once per opened unit, on its entry price) ----
        sold, bought, slippage, pos = 0.0, 0.0, 0.0, {}
        for _, scrip, price, side, qty in orderbook:
            signed = qty if side == 'BUY' else -qty
            cur = pos.get(scrip, 0)
            opened = qty - min(abs(cur), qty) if cur * signed < 0 else qty   # units not closing an existing position
            slippage += bt.Cal_slipage(price) * opened
            if side == 'BUY': bought += price * qty
            else:             sold   += price * qty
            pos[scrip] = cur + signed
        gross = sold - bought
        
        return [
            code, bt.index, start_time, end_time, sell_om, buy_om, wing_width, max_rolls, cap_action,
            bt.current_week_dates[0].date(), bt.current_week_dates[-1].date(), bt.from_dte, bt.to_dte, len(bt.current_week_dates),
            entry_time, future_price, sell_ce_scrip, sell_ce_price, buy_ce_scrip, buy_ce_price, round(sell_ce_price - buy_ce_price, 2),
            len(rolls), capped, ' | '.join(rolls), f"{int(sold_strike)}CE", sold_qnty,
            exit_dt, ' | '.join(squareoff),
            len(orderbook), sum(q for *_, q in orderbook), round(sold, 2), round(bought, 2), round(gross, 2), round(slippage, 2), round(gross - slippage, 2),
        ]
        
    except Exception as e:
        print(e, [bt.index, bt.current_week_dates[0].date(), bt.current_week_dates[-1].date(), start_time, end_time, sell_om, buy_om, wing_width, max_rolls, cap_action])
        return

In [ ]:
for row_idx in range(len(meta_data)):

    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, from_dte, to_dte, from_date, to_date, start_time, end_time, week_lists = get_meta_row_data(meta_row, pickle_path, weekly=True)

            log_cols = ('P_Strategy/P_Index/P_StartTime/P_EndTime/P_SellOM/P_BuyOM/P_WingWidth/P_MaxRolls/P_CapAction/Start.Date/End.Date/Start.DTE/End.DTE/DayCount/EntryTime/Future/Sell.Strike/Sell.Price/Buy.Strike/Buy.Price/Net.Credit/Rolls/Capped/Roll.Log/Last.Sold.Strike/Max.Qnty/ExitTime/Squareoff/Orders/Lots.Traded/Premium.Sold/Premium.Bought/Gross.PNL/Slippage/Total.PNL').split('/')

            for week_dates in week_lists:
                from_date = week_dates[0]
                to_date = week_dates[-1]

                file_name = f"{index} {week_dates[0].date()} {week_dates[-1].date()} {from_dte}-{to_dte} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):

                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")

                    wbt = WeeklyBacktest(pickle_path, index, week_dates, from_dte, to_dte, start_time, end_time)
                    wbt._slipage_rate = 0.005   # 0.5% slippage, this strategy only
                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)

                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [CE_BUTTERFLY(wbt, row['entry_time'], row['exit_time'], row['sell_om'], row['buy_om'], row['wing_width'], row['max_rolls'], row['cap_action']) for idx, row in tqdm(chunk_parameter.iterrows(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)
                        
                        del chunk
                        del chunk_parameter
                        gc.collect()

                    del wbt
                    gc.collect()
                    
                    t2 = datetime.datetime.now()
                    print(t2-t1)

        except Exception as e:
            input(str(e))